In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [6]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print(f"Train Shape: {train_df.shape}")
print(f"Test Shape: {test_df.shape}")

# Check for missing values
print("\nMissing values in Train set:")
print(train_df.isnull().sum())

print("\nMissing values in Test set:")
print(test_df.isnull().sum())

# Handle missing data if any (fill with empty string)
text_columns = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in text_columns:
    train_df[col] = train_df[col].fillna("")
    test_df[col] = test_df[col].fillna("")

Train Shape: (2000, 8)
Test Shape: (500, 7)

Missing values in Train set:
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Missing values in Test set:
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64


In [8]:
import string
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# 1. Frequency distribution of correct answers
counts = df['answer'].value_counts()
most_freq = counts.max()
least_freq = counts.min()
sum_most_least = most_freq + least_freq


Q10: TF-IDF Pipeline MAP@3: 0.31192
Q1: Distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Sum of most and least: 814
Q2: Prompt Vocab Size: 859
Q3: Row 1 remaining words: 13
Q4: Total unique features: 2762
Q5: Similarity score: 0.2328
Q6: Top similarity match percentage: 13.70%
Q7: MAP@3 for C from CAB: 1.0
Q8: MAP@3 for B from DBE: 0.5
Q9: Majority Class MAP@3: 0.42125
Q10: TF-IDF Pipeline MAP@3: 0.31192


In [ ]:

# 2. Vocabulary size of cleaned prompt column
def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

all_words = set()
for p in df['prompt']:
    all_words.update(clean_text(p))
vocab_size_prompt = len(all_words)


In [ ]:

# 3. Filter row ID 1 cleaned prompt with sklearn stop words
row1_prompt_words = clean_text(df.iloc[0]['prompt'])
row1_filtered = [w for w in row1_prompt_words if w not in ENGLISH_STOP_WORDS]
row1_filtered_count = len(row1_filtered)


In [ ]:

# 4. Fit TfidfVectorizer on all combined text
combined_texts = []
for idx, row in df.iterrows():
    combined_texts.append(str(row['prompt']))
    for opt in ['A', 'B', 'C', 'D', 'E']:
        combined_texts.append(str(row[opt]))

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)
total_features = len(vectorizer.vocabulary_)


In [ ]:

# 5. Cosine similarity for Row ID 1 prompt and option A
p_vector = vectorizer.transform([str(df.iloc[0]['prompt'])])
a_vector = vectorizer.transform([str(df.iloc[0]['A'])])
row1_sim = cosine_similarity(p_vector, a_vector)[0][0]



In [ ]:
# 6. Percentage where highest similarity matches correct answer
match_count = 0
tfidf_pipeline_preds = []
for idx, row in df.iterrows():
    p_vec = vectorizer.transform([str(row['prompt'])])
    sims = {}
    for opt in ['A', 'B', 'C', 'D', 'E']:
        o_vec = vectorizer.transform([str(row[opt])])
        sims[opt] = cosine_similarity(p_vec, o_vec)[0][0]
    
    sorted_opts = sorted(sims.items(), key=lambda x: x[1], reverse=True)
    top_opt = sorted_opts[0][0]
    if top_opt == row['answer']:
        match_count += 1
    tfidf_pipeline_preds.append([o[0] for o in sorted_opts[:3]])

pct_highest_match = (match_count / len(df)) * 100


In [ ]:

# 7 & 8 MAP@3 Helper Function
def apk(actual, predicted, k=3):
    if actual not in predicted[:k]:
        return 0.0
    return 1.0 / (predicted[:k].index(actual) + 1.0)

map3_q7 = apk('C', ['C', 'A', 'B'])
map3_q8 = apk('B', ['D', 'B', 'E'])


In [ ]:

# 9. Majority Class Baseline
top_3_labels = list(counts.index[:3])
majority_map3 = np.mean([apk(row['answer'], top_3_labels) for idx, row in df.iterrows()])

# 10. TF-IDF Pipeline Baseline

tfidf_pipeline_map3 = np.mean([apk(actual, preds) for actual, preds in zip(df['answer'], tfidf_pipeline_preds)])

print(f"Q10: TF-IDF Pipeline MAP@3: {tfidf_pipeline_map3:.5f}")

print(f"Q1: Distribution:\n{counts}\nSum of most and least: {sum_most_least}")
print(f"Q2: Prompt Vocab Size: {vocab_size_prompt}")
print(f"Q3: Row 1 remaining words: {row1_filtered_count}")
print(f"Q4: Total unique features: {total_features}")
print(f"Q5: Similarity score: {row1_sim:.4f}")
print(f"Q6: Top similarity match percentage: {pct_highest_match:.2f}%")
print(f"Q7: MAP@3 for C from CAB: {map3_q7}")
print(f"Q8: MAP@3 for B from DBE: {map3_q8}")
print(f"Q9: Majority Class MAP@3: {majority_map3:.5f}")
print(f"Q10: TF-IDF Pipeline MAP@3: {tfidf_pipeline_map3:.5f}")